# Google → Meta (Muse) data-sharing emails: SUPRA and non-SUPRA panels

**Goal.** Track Google's confirmation emails that a panellist shared Google Account data with Meta (Muse Gmail access). Expected subject:

> *You shared some Google Account data with Meta*

Names may be redacted to `__` (e.g. `You shared some __ Account data with __`), so the match is anchored on the fixed words only.

**Steps**
1. Check whether `google.com` is a sender domain in each source, both SUPRA and non-SUPRA.
2. For sources where it is, search Google-sent subjects for the *"shared some … data with …"* pattern and track it weekly.

**Sources (`SHARE_NIQ.SRS`)**

| Group | View | Date / mailbox |
|---|---|---|
| SUPRA | `SUPRA_QUALITY_MERCHANT_EMAIL_HEADER` | via `SUPRA_QUALITY_MERCHANT_EMAILS` on `SUBJECT_ID` |
| SUPRA | `SUPRA_NON_QUALITY_MERCHANT_EMAIL_HEADER` | via `SUPRA_NON_QUALITY_MERCHANT_EMAILS` on `SUBJECT_ID` |
| non-SUPRA | `EMAIL_HEADER`, `EMAIL_HEADER_FTA`, `EMAIL_HEADER_FTA_BANKING` | `EMAIL_SENT_DATE` in-table; no mailbox |
| non-SUPRA | `US_MERCHANT_EMAILS_FLATTEN` | `EMAIL_SENT_DATE`, `MAILBOX_ID` in-table |

**Status as of 2026-10-06 (checked ad hoc):**
- SUPRA: `google` is **not** a sender domain in either panel. Quality has 22 merchant domains; non-quality has invisalign and 360incentives.
- non-SUPRA `EMAIL_HEADER`: `google.com` **is** present (about 1.7M rows since 2026-09-01). These are all commerce senders (`googleplay-noreply`, `googleone-noreply`, `googlestore-noreply`, Google Cloud payments). There were **no** *"shared some … data with …"* subjects in September 2026.

The non-SUPRA tables are large, so they're scanned from `START_DATE` only. Widen it if needed.

In [ ]:
import os
import pandas as pd
import snowflake.connector

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", 120)

# Uses a named connection from ~/.snowflake/connections.toml (same pattern as srs-evals).
conn = snowflake.connector.connect(
    connection_name=os.getenv("SNOWFLAKE_CONNECTION", "default"),
)

def q(sql: str) -> pd.DataFrame:
    df = conn.cursor().execute(sql).fetch_pandas_all()
    df.columns = [c.lower() for c in df.columns]
    return df

S = "SHARE_NIQ.SRS"
START_DATE = "2026-01-01"  # lower bound on email_sent_date for every scan that has a date

# Each source exposes h.domain / h.top_level_domain / h.subject, plus:
#   presence_from: cheapest FROM for the domain check (SUPRA headers carry no date, so no join needed)
#   presence_date: date column for the domain check, or None
#   emails_from:   email-grain FROM (SUPRA joins header -> emails on SUBJECT_ID)
#   sent / sender / mailbox: column expressions on emails_from
_hdr_sender = "h.local_part || '@' || h.domain || '.' || h.top_level_domain"
SOURCES = [
    dict(group="SUPRA", name="SUPRA quality",
         presence_from=f"{S}.SUPRA_QUALITY_MERCHANT_EMAIL_HEADER h", presence_date=None,
         emails_from=f"{S}.SUPRA_QUALITY_MERCHANT_EMAIL_HEADER h JOIN {S}.SUPRA_QUALITY_MERCHANT_EMAILS e ON e.subject_id = h.subject_id",
         sent="e.email_sent_date", sender="h.from_addr", mailbox="e.mailbox_id"),
    dict(group="SUPRA", name="SUPRA non-quality",
         presence_from=f"{S}.SUPRA_NON_QUALITY_MERCHANT_EMAIL_HEADER h", presence_date=None,
         emails_from=f"{S}.SUPRA_NON_QUALITY_MERCHANT_EMAIL_HEADER h JOIN {S}.SUPRA_NON_QUALITY_MERCHANT_EMAILS e ON e.subject_id = h.subject_id",
         sent="e.email_sent_date", sender="h.from_addr", mailbox="e.mailbox_id"),
    *[
        dict(group="non-SUPRA", name=t,
             presence_from=f"{S}.{t} h", presence_date="h.email_sent_date",
             emails_from=f"{S}.{t} h", sent="h.email_sent_date", sender=_hdr_sender, mailbox="NULL")
        for t in ("EMAIL_HEADER", "EMAIL_HEADER_FTA", "EMAIL_HEADER_FTA_BANKING")
    ],
    dict(group="non-SUPRA", name="US_MERCHANT_EMAILS_FLATTEN",
         presence_from=f"{S}.US_MERCHANT_EMAILS_FLATTEN h", presence_date="h.email_sent_date",
         emails_from=f"{S}.US_MERCHANT_EMAILS_FLATTEN h", sent="h.email_sent_date", sender="h.from_addr", mailbox="h.mailbox_id"),
]

IS_GOOGLE = "(LOWER(h.domain) = 'google' AND LOWER(h.top_level_domain) = 'com')"

# "You shared some Google Account data with Meta"; names may be redacted to __, so match the fixed words only.
SHARED_DATA = "(h.subject ILIKE '%shared some%data with%')"

## 1. Is `google.com` a sender domain?

One query per source, so a slow non-SUPRA table doesn't block the rest. `google_rows` counts rows with domain `google` / TLD `com`.

In [ ]:
rows = []
for src in SOURCES:
    where = f"WHERE {src['presence_date']} >= '{START_DATE}'" if src["presence_date"] else ""
    r = q(f"""
        SELECT COUNT(*) AS total_rows, COUNT_IF({IS_GOOGLE}) AS google_rows
        FROM {src['presence_from']} {where}
    """).iloc[0]
    rows.append(dict(group=src["group"], source=src["name"],
                     window=f">= {START_DATE}" if src["presence_date"] else "all (no date on header)",
                     total_rows=r.total_rows, google_rows=r.google_rows))

presence = pd.DataFrame(rows).assign(google_present=lambda d: d.google_rows > 0)
presence

In [ ]:
# Which Google senders appear, for the sources where Google is present
google_sources = [s for s in SOURCES if s["name"] in set(presence.loc[presence.google_present, "source"])]
print("Google present in:", [s["name"] for s in google_sources] or "none")

senders = []
for src in google_sources:
    senders.append(q(f"""
        SELECT '{src['name']}' AS source, LOWER({src['sender']}) AS sender, COUNT(*) AS n_emails,
               MIN({src['sent']}) AS first_seen, MAX({src['sent']}) AS last_seen
        FROM {src['emails_from']}
        WHERE {IS_GOOGLE} AND {src['sent']} >= '{START_DATE}'
        GROUP BY 1, 2
    """))
pd.concat(senders).sort_values("n_emails", ascending=False) if senders else None

## 2. *"You shared some … data with …"* subjects from Google

Only runs on sources where Step 1 found Google. Pattern: `subject ILIKE '%shared some%data with%'`.

In [ ]:
matches = []
for src in google_sources:
    matches.append(q(f"""
        SELECT '{src['name']}' AS source, LOWER({src['sender']}) AS sender, h.subject,
               COUNT(*) AS n_emails, MIN({src['sent']}) AS first_seen, MAX({src['sent']}) AS last_seen
        FROM {src['emails_from']}
        WHERE {IS_GOOGLE} AND {SHARED_DATA} AND {src['sent']} >= '{START_DATE}'
        GROUP BY 1, 2, 3
    """))
matches = pd.concat(matches) if matches else pd.DataFrame()
print(f"{len(matches)} matching subject variants" if not matches.empty else "No 'shared some … data with …' subjects from Google.")
matches.sort_values("n_emails", ascending=False) if not matches.empty else None

In [ ]:
# Weekly tracker for sources with matches. Mailboxes are only available for SUPRA and US_MERCHANT_EMAILS_FLATTEN.
match_sources = [s for s in google_sources if not matches.empty and s["name"] in set(matches.source)]
weekly = []
for src in match_sources:
    weekly.append(q(f"""
        SELECT '{src['name']}' AS source, DATE_TRUNC('week', {src['sent']}) AS week,
               COUNT(*) AS n_emails, COUNT(DISTINCT {src['mailbox']}) AS n_mailboxes
        FROM {src['emails_from']}
        WHERE {IS_GOOGLE} AND {SHARED_DATA} AND {src['sent']} >= '{START_DATE}'
        GROUP BY 1, 2
    """))

if weekly:
    weekly = pd.concat(weekly).sort_values(["source", "week"])
    weekly.pivot_table(index="week", columns="source", values="n_emails", aggfunc="sum").fillna(0).plot(
        title="Google 'You shared some … data with …' emails per week", figsize=(10, 4))
    display(weekly)
else:
    print("Nothing to track yet.")